### Configuration and imports

In [1]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src/legal_rag").is_dir():
    raise RuntimeError("Open this notebook with MyLLMAgent as the working directory.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from legal_rag.training.dataset import (
    file_hash,
    read_jsonl,
    prepare_dataset,
)

from legal_rag.training.tokenization import (
    balance_encoded,
    pad_batch,
    prompt_text,
)

SOURCE_ROOT = PROJECT_ROOT.parent / "legal-rag/data/processed"
DATA_DIR = PROJECT_ROOT / "data/training/legal-qa-pilot-v1"
TOKENIZER_DIR = PROJECT_ROOT / "data/training/qwen3-4b-tokenizer"

MODEL_ID = "Qwen/Qwen3-4B"
MODEL_REVISION = "1cfa9a7208912126459214e8b04321603b3df60c"

DOMAINS = (
    "civil",
    "criminal",
    "administrative",
    "intellectual_property",
)

MAX_LENGTH = 1024
SEED = 42

RUN_GPU = True

# FIRST GPU run: "smoke"
# After the smoke run succeeds: restart the kernel and use "train".
RUN_MODE = "train"

print("Source directory: ", SOURCE_ROOT)
print("Prepared dataset: ", DATA_DIR)
print("GPU work enabled; ", RUN_GPU)

Source directory:  /home/layke/Documents/legal-rag/data/processed
Prepared dataset:  /home/layke/Documents/MyLLMAgent/data/training/legal-qa-pilot-v1
GPU work enabled;  True


### Prepare or verify the dataset

In [2]:
if not DATA_DIR.exists():
    prepare_dataset(
        SOURCE_ROOT,
        DATA_DIR,
        seed = SEED,
        per_domain={
            "train": 1000,
            "validation": 100,
            "test": 100, 
        },
        max_source_chars=2400,
    )

manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))

for filename, expected_path in manifest["files"].items():
    actual_hash = file_hash(DATA_DIR / filename)

    if actual_hash != expected_path:
        raise RuntimeError(f"Dataset changed: {filename}. "
                           "Prepare a new dataset version instead of modifying this one.")

raw_splits = {
    split: list(read_jsonl(DATA_DIR / f"{split}.jsonl"))
    for split in ("train", "validation", "test")
}

seen_groups = {}
seen_questions = set()

for split, examples in raw_splits.items():
    for example in examples:
        group = example["group_id"]

        if seen_groups.setdefault(group, split) != split:
            raise RuntimeError("A source document group crosses datset splites")

        if example["question_hash"] in seen_questions:
            raise RuntimeError("A duplicate question exists in the prepared dataset")

        seen_questions.add(example["question_hash"])

    domain_counts = {
        domain: sum(example["domain"] == domain for example in examples)
        for domain in DOMAINS
    }

    print(split, domain_counts)

print("Dataset hashes and split checks passed.")

train {'civil': 1000, 'criminal': 1000, 'administrative': 1000, 'intellectual_property': 1000}
validation {'civil': 100, 'criminal': 100, 'administrative': 100, 'intellectual_property': 100}
test {'civil': 100, 'criminal': 100, 'administrative': 100, 'intellectual_property': 100}
Dataset hashes and split checks passed.


### Inspect training examples

In [3]:
for domain in DOMAINS:
    example = next(
        example for example in raw_splits["train"] if example["domain"] == domain
    )

    print("\n" + "=" * 80)
    print("DOMAIN: ", domain)
    print("EXAMPLE ID: ", example["id"])
    print("DOCUMENT ID: ", example["document_id"])

    print("\nQUESTION: ")
    print(example["question"])

    print("ANSWER: ")
    print(example["answer"])


DOMAIN:  civil
EXAMPLE ID:  82ffa39423e308d0f3efd79b
DOCUMENT ID:  서울중앙지방법원-2017가단5079796

QUESTION: 
언론사와 기자의 보도로 인해 개인이 심각한 정신적 고통을 입었을 때, 언론사와 기자가 손해배상을 해야 하는 조건은 무엇인가요?
ANSWER: 
언론사와 기자는 보도로 인해 개별인이 입은 심각한 정신적 고통에 대해 손해배상을 해야 합니다. 언론보도에 허위사실이 포함되어 있어 그로 인해 피해자가 정신적 고통을 입었다는 것을 증명하면, 언론사와 기자는 손해를 배상할 의무가 있습니다. 또한, 손해배상 범위를 산정할 때는 피해자의 인터뷰 또는 행동이 악성 댓글 등의 원인이 되었는지 여부도 함께 고려합니다.

DOMAIN:  criminal
EXAMPLE ID:  5b55dbc422ed753fd2b730ea
DOCUMENT ID:  criminal::judgment::189019

QUESTION: 
재크나이프가 폭력행위 등 처벌에 관한 법률 제7조의 흉기로 인정되지 않은 이유는 무엇인가요?
ANSWER: 
재크나이프는 그 자체만으로 폭력행위 등 처벌에 관한 법률 제7조에서 정한 범죄에 공용될 우려가 있는 흉기로 볼 수 없습니다. 해당 법률은 정당한 이유 없이 범죄에 공용될 우려가 있는 흉기나 위험한 물건을 휴대한 경우 처벌을 규정하고 있습니다. 그러나, 피고인이 소지한 재크나이프가 실제로 범죄에 사용될 우려가 있다는 객관적 증거가 기록에 존재하지 않았고, 단순 소지 사실만으로 흉기로 추정할 수 없다는 이유로 무죄가 선고되었습니다.

DOMAIN:  administrative
EXAMPLE ID:  39b1f95fc96c7a344e0963ca
DOCUMENT ID:  administrative::constitutional_decision::21576

QUESTION: 
청구인이 서울 중구 신당3동 동사무소 직원의 처벌을 구하는 취지로 헌법소원심판을 다시 청구했지만, 왜 각하되었나요?
AN

### Tokenize and inspect the training labels

In [4]:
from transformers import AutoTokenizer

if TOKENIZER_DIR.is_dir():
    revision_file = TOKENIZER_DIR / "revision.txt"

    if (not revision_file.is_file() or revision_file.read_text().strip() != MODEL_REVISION):
        raise RuntimeError("The local tokenizer revision does not math MODEL_REVISION")

    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, local_files_only=True)

else:
    # Downloads tokenizer files only, not model weights.
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
    )

tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenized = {}
token_report = {}
selected_raw = {}

for split, examples in raw_splits.items():
    encoded, report = balance_encoded(
        examples,
        tokenizer,
        max_length=MAX_LENGTH,
        seed=SEED,
        domains=DOMAINS,
    )

    tokenized[split] = encoded
    token_report[split] = report

    examples_by_id = {
        example["id"]: example
        for example in examples
    }

    # Keep original examples aligned with their tokenized versions.
    selected_raw[split] = [
        examples_by_id[example_id]
        for example_id in report["selected_ids"]
    ]

    printable_report = {
        key: value
        for key, value in report.items()
        if key != "selected_ids"
    }

    print(split, printable_report)

    for example in encoded:
        supervised_tokens = sum(
            label != -100
            for label in example["labels"]
        )

        assert 0 < supervised_tokens < len(example["labels"])
        assert len(example["input_ids"]) <= MAX_LENGTH


# Check the actual examples selected AFTER tokenization.
# This section must be outside the loop above.
from itertools import combinations

for field in ("id", "group_id", "question_hash"):
    split_sets = {
        split: {
            example[field]
            for example in examples
        }
        for split, examples in selected_raw.items()
    }

    for first, second in combinations(split_sets, 2):
        overlap = split_sets[first] & split_sets[second]

        if overlap:
            raise RuntimeError(
                f"Data leakage: {first} and {second} share "
                f"{len(overlap)} values of {field}."
            )

print("\nTokenized splits are disjoint.")

for split, examples in tokenized.items():
    print(f"{split}: {len(examples):,} examples")

print("\nPreparation complete.")
print("The next call stops unless RUN_GPU is enabled.")

/home/layke/anaconda3/envs/legal-qlora/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train {'max_length': 1024, 'overlength_by_domain': {'civil': 722, 'criminal': 488, 'intellectual_property': 841, 'administrative': 731}, 'eligible_by_domain': {'civil': 278, 'criminal': 512, 'administrative': 269, 'intellectual_property': 159}, 'selected_per_domain': 159}
validation {'max_length': 1024, 'overlength_by_domain': {'intellectual_property': 82, 'criminal': 53, 'civil': 76, 'administrative': 76}, 'eligible_by_domain': {'civil': 24, 'criminal': 47, 'administrative': 24, 'intellectual_property': 18}, 'selected_per_domain': 18}
test {'max_length': 1024, 'overlength_by_domain': {'intellectual_property': 86, 'administrative': 74, 'criminal': 51, 'civil': 76}, 'eligible_by_domain': {'civil': 24, 'criminal': 49, 'administrative': 26, 'intellectual_property': 14}, 'selected_per_domain': 14}

Tokenized splits are disjoint.
train: 636 examples
validation: 72 examples
test: 56 examples

Preparation complete.
The next call stops unless RUN_GPU is enabled.


### Stop before GPU work

In [5]:
if not RUN_GPU:
    raise RuntimeError(
        "Preparation only, After testing finishes, set RUN_GPU=True and rerun."
    )
if RUN_MODE not in {"smoke", "train"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'train'.")

import importlib.metadata

expected_versions = {
    "torch": "2.8.0",
    "transformers": "4.57.1",
    "peft": "0.17.1",
    "bitsandbytes": "0.48.1",
    "accelerate": "1.10.1",
}

for package, expected_version in expected_versions.items():
    installed_version = (importlib.metadata.version(package).split("+")[0])

    if installed_version != expected_version:
        raise RuntimeError(
            f"Use the legal-qlora environment: "
            f"{package} {installed_version} != {expected_version}"
        )

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Check the NVIDIA driver and training environment."
    )

free_memory, total_memory = torch.cuda.mem_get_info()

print("GPU:", torch.cuda.get_device_name(0))
print(f"Free memoryL {free_memory / 2**30:.2f} GiB")
print(f"Total memory; {total_memory / 2**30:.2f} GiB")

if free_memory < 6 * 2**30:
    raise RuntimeError("Less than 6 GiB is free. Finish GPU testing and unload unused serving models first.")

print("Initial memory check passed.")
print("The smoke run still needs to establish actual memory fit.")

GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Free memoryL 7.51 GiB
Total memory; 7.62 GiB
Initial memory check passed.
The smoke run still needs to establish actual memory fit.


### Load Qwen3-4B and attach the QLoRA adapter

In [6]:
if not RUN_GPU:
    raise RuntimeError("GPU work is disabled.")

from datetime import datetime, timezone

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

set_seed(SEED)

compute_dtype = (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    attn_implementation="sdpa",
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    }
)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

run_name = f"{timestamp}-{RUN_MODE}"

OUTPUT_DIR = PROJECT_ROOT / "models/adapters/qwen3-4b-legal" / run_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

run_record = {
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "mode": RUN_MODE,
    "seed": SEED,
    "dataset_manifest_sha256": file_hash(DATA_DIR / "manifest.json"),
    "dataset_files": manifest["files"],
    "tokenization": token_report,
    "versions": {package: importlib.metadata.version(package) for package in expected_versions},
    "gpu": torch.cuda.get_device_name(0),
    "dtype": str(compute_dtype),
}

(OUTPUT_DIR / "run.json").write_text(json.dumps(run_record, indent=2) + "\n", encoding="utf-8")

print("Run output: ", OUTPUT_DIR)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:07<00:00,  2.49s/it]


trainable params: 16,515,072 || all params: 4,038,983,168 || trainable%: 0.4089
Run output:  /home/layke/Documents/MyLLMAgent/models/adapters/qwen3-4b-legal/20260924T032523665089Z-train


In [7]:
recovery_dir = OUTPUT_DIR / "adapter-interrupted"

model.save_pretrained(
    str(recovery_dir),
    safe_serialization=True,
)
tokenizer.save_pretrained(str(recovery_dir))

print("Saved interrupted adapter:", recovery_dir)

Saved interrupted adapter: /home/layke/Documents/MyLLMAgent/models/adapters/qwen3-4b-legal/20260924T032523665089Z-train/adapter-interrupted


### Baseline validation, training, and saving

In [8]:
if not RUN_GPU:
    raise RuntimeError("GPU work is disabled.")

import gc
import json
from pathlib import Path

import torch
from transformers import Trainer, TrainingArguments


# None starts a fresh run.
# To resume, set this to an existing checkpoint directory.
RESUME_FROM_CHECKPOINT = None


def collate(features):
    padded = pad_batch(
        features,
        pad_token_id=tokenizer.pad_token_id,
    )

    return {
        key: torch.tensor(value, dtype=torch.long)
        for key, value in padded.items()
    }


def domain_dataset(split, domain):
    return [
        encoded
        for encoded, original in zip(
            tokenized[split],
            selected_raw[split],
        )
        if original["domain"] == domain
    ]


def release_cached_memory():
    gc.collect()
    torch.cuda.empty_cache()


def save_json(filename, payload):
    path = OUTPUT_DIR / filename
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )


train_data = tokenized["train"]
validation_data = tokenized["validation"]

if RUN_MODE == "smoke":
    train_data = [
        example
        for domain in DOMAINS
        for example in domain_dataset("train", domain)[:2]
    ]

    validation_data = [
        example
        for domain in DOMAINS
        for example in domain_dataset("validation", domain)[:1]
    ]

if not train_data or not validation_data:
    raise RuntimeError("Training and validation datasets must be nonempty.")

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if RESUME_FROM_CHECKPOINT is not None:
    RESUME_FROM_CHECKPOINT = str(
        Path(RESUME_FROM_CHECKPOINT).resolve()
    )

    if not Path(RESUME_FROM_CHECKPOINT).is_dir():
        raise FileNotFoundError(RESUME_FROM_CHECKPOINT)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=(
        1 if RUN_MODE == "smoke" else 16
    ),

    num_train_epochs=1,
    max_steps=1 if RUN_MODE == "smoke" else -1,

    learning_rate=1e-4,
    lr_scheduler_type=(
        "constant" if RUN_MODE == "smoke" else "cosine"
    ),
    warmup_ratio=0.0 if RUN_MODE == "smoke" else 0.03,

    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,

    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    # Evaluate explicitly before and after training.
    # This prevents automatic evaluation from interrupting training.
    eval_strategy="no",
    prediction_loss_only=True,

    # Full checkpoints allow interrupted training to resume.
    save_strategy="steps",
    save_steps=1,
    save_total_limit=2,
    save_only_model=False,

    logging_steps=1 if RUN_MODE == "smoke" else 10,
    report_to="none",

    seed=SEED,
    data_seed=SEED,

    dataloader_num_workers=0,
    dataloader_pin_memory=False,
)

model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=validation_data,
    data_collator=collate,
    processing_class=tokenizer,
)

(OUTPUT_DIR / "training_args.json").write_text(
    training_args.to_json_string(),
    encoding="utf-8",
)

metrics = {
    "mode": RUN_MODE,
    "train_examples": len(train_data),
    "validation_examples": len(validation_data),
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
}

# ------------------------------------------------------------
# 1. BASELINE VALIDATION
# ------------------------------------------------------------

print("Evaluating the base model with the adapter disabled...")

release_cached_memory()

try:
    with model.disable_adapter():
        baseline_validation = trainer.evaluate(
            metric_key_prefix="base_validation"
        )

    metrics["baseline_validation"] = baseline_validation
    print(json.dumps(baseline_validation, indent=2))

except torch.cuda.OutOfMemoryError:
    metrics["baseline_validation"] = {
        "status": "failed",
        "reason": "CUDA out of memory",
    }

    save_json("metrics.json", metrics)

    # Stop here rather than starting training after a failed baseline.
    raise

save_json("metrics.json", metrics)

# ------------------------------------------------------------
# 2. TRAINING
# ------------------------------------------------------------

release_cached_memory()
torch.cuda.reset_peak_memory_stats()

print("Starting training...")

try:
    train_result = trainer.train(
        resume_from_checkpoint=RESUME_FROM_CHECKPOINT
    )

except torch.cuda.OutOfMemoryError:
    metrics["training"] = {
        "status": "interrupted",
        "reason": "CUDA out of memory",
        "global_step": trainer.state.global_step,
    }

    save_json("metrics.json", metrics)

    checkpoint_root = OUTPUT_DIR / "checkpoints"
    checkpoints = sorted(
        checkpoint_root.glob("checkpoint-*"),
        key=lambda path: int(path.name.rsplit("-", 1)[-1]),
    )

    if checkpoints:
        print("Latest resumable checkpoint:", checkpoints[-1])
    else:
        print("No periodic checkpoint was saved before the failure.")

    print("Retrying training with a shorter sequence length...")

    model.zero_grad(set_to_none=True)

    trainer.optimizer = None
    trainer.lr_scheduler = None
    trainer.accelerator.free_memory()

    del trainer
    release_cached_memory()

    retry_max_length = 768

    def retry_collate(features):
        truncated = [
            {
                "input_ids": feature["input_ids"][-retry_max_length:],
                "labels": feature["labels"][-retry_max_length:],
            }
            for feature in features
        ]

        padded = pad_batch(
            truncated,
            pad_token_id=tokenizer.pad_token_id,
        )

        return {
            key: torch.tensor(value, dtype=torch.long)
            for key, value in padded.items()
        }

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=validation_data,
        data_collator=retry_collate,
        processing_class=tokenizer,
    )

    try:
        train_result = trainer.train(
            resume_from_checkpoint=RESUME_FROM_CHECKPOINT
        )
    except torch.cuda.OutOfMemoryError:
        save_json("metrics.json", metrics)
        metrics["training"]["retry_status"] = "failed"
        metrics["training"]["retry_reason"] = (
            "CUDA out of memory after retrying with "
            f"max sequence length {retry_max_length}."
        )
        save_json("metrics.json", metrics)

        raise RuntimeError(
            "Training failed because the GPU ran out of memory. "
            "Reduce the sequence length further, for example to 512."
        ) from None

metrics["training"] = train_result.metrics
metrics["training_peak_allocated_gib"] = (
    torch.cuda.max_memory_allocated() / 2**30
)
metrics["training_peak_reserved_gib"] = (
    torch.cuda.max_memory_reserved() / 2**30
)

# ------------------------------------------------------------
# 3. SAVE BEFORE POST-TRAINING EVALUATION
# ------------------------------------------------------------

print("Training finished. Saving the adapter immediately...")

adapter_dir = OUTPUT_DIR / "adapter"

trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

trainer.save_metrics("train", train_result.metrics)

# Trainer metadata; resumable optimizer/scheduler state is stored
# in the periodic checkpoint directories.
trainer.save_state()

metrics["adapter_directory"] = str(adapter_dir)
save_json("metrics.json", metrics)

print("Adapter saved:", adapter_dir)
print("Periodic checkpoints:", OUTPUT_DIR / "checkpoints")

# Release optimizer and scheduler references before final evaluation.
# Resume future training from a checkpoint using a newly built trainer.
trainer.optimizer = None
trainer.lr_scheduler = None

# Accelerate also keeps references to prepared optimizers/schedulers.
trainer.accelerator.free_memory()

model.zero_grad(set_to_none=True)
release_cached_memory()

# Use a fresh Trainer for evaluation only.
del trainer
release_cached_memory()

evaluation_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "evaluation"),
    per_device_eval_batch_size=1,
    prediction_loss_only=True,
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    report_to="none",
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
)

evaluation_trainer = Trainer(
    model=model,
    args=evaluation_args,
    eval_dataset=validation_data,
    data_collator=collate,
    processing_class=tokenizer,
)

# ------------------------------------------------------------
# 4. POST-TRAINING VALIDATION
# ------------------------------------------------------------

print("Evaluating the saved adapter...")

try:
    trained_validation = evaluation_trainer.evaluate(
        metric_key_prefix="adapter_validation"
    )

    metrics["adapter_validation"] = trained_validation
    print(json.dumps(trained_validation, indent=2))

except torch.cuda.OutOfMemoryError:
    metrics["adapter_validation"] = {
        "status": "failed",
        "reason": "CUDA out of memory",
    }

    print(
        "Post-training validation ran out of memory. "
        "The trained adapter has already been saved."
    )
    print("Saved adapter:", adapter_dir)

finally:
    save_json("metrics.json", metrics)

print("\nRun results:")
print(json.dumps(metrics, indent=2))

Evaluating the base model with the adapter disabled...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


{
  "base_validation_loss": 1.4666966199874878,
  "base_validation_model_preparation_time": 0.0048,
  "base_validation_runtime": 32.9828,
  "base_validation_samples_per_second": 2.183,
  "base_validation_steps_per_second": 2.183
}
Starting training...


Step,Training Loss
10,1.180200
20,0.857700
30,0.795100
40,0.857100


Training finished. Saving the adapter immediately...
Adapter saved: /home/layke/Documents/MyLLMAgent/models/adapters/qwen3-4b-legal/20260924T032523665089Z-train/adapter
Periodic checkpoints: /home/layke/Documents/MyLLMAgent/models/adapters/qwen3-4b-legal/20260924T032523665089Z-train/checkpoints
Evaluating the saved adapter...


{
  "adapter_validation_loss": 0.754889965057373,
  "adapter_validation_model_preparation_time": 0.0048,
  "adapter_validation_runtime": 39.6876,
  "adapter_validation_samples_per_second": 1.814,
  "adapter_validation_steps_per_second": 1.814
}

Run results:
{
  "mode": "train",
  "train_examples": 636,
  "validation_examples": 72,
  "resume_from_checkpoint": null,
  "baseline_validation": {
    "base_validation_loss": 1.4666966199874878,
    "base_validation_model_preparation_time": 0.0048,
    "base_validation_runtime": 32.9828,
    "base_validation_samples_per_second": 2.183,
    "base_validation_steps_per_second": 2.183
  },
  "training": {
    "train_runtime": 1129.3239,
    "train_samples_per_second": 0.563,
    "train_steps_per_second": 0.035,
    "total_flos": 1.1121427891863552e+16,
    "train_loss": 0.9225190877914429,
    "epoch": 1.0
  },
  "training_peak_allocated_gib": 6.20635461807251,
  "training_peak_reserved_gib": 7.4609375,
  "adapter_directory": "/home/layke/Documen

### Compare the base model and adapter on held-out questions

In [9]:
if not RUN_GPU:
    raise RuntimeError("GPU work is disabled.")

if RUN_MODE == "smoke":
    print(
        "Skipping the held-out comparison for smoke mode. "
        "Run the full pilot successfully first."
    )

else:
    import gc
    import json
    from contextlib import nullcontext

    import torch
    from transformers import set_seed

    if "evaluation_trainer" not in globals():
        raise RuntimeError(
            "evaluation_trainer is unavailable. "
            "Complete the updated training and saving cell first."
        )

    adapter_dir = OUTPUT_DIR / "adapter"

    if not (adapter_dir / "adapter_config.json").is_file():
        raise RuntimeError(
            f"No completed adapter export found at {adapter_dir}."
        )

    comparison_dir = OUTPUT_DIR / "comparison"
    comparison_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = comparison_dir / "test_metrics.json"
    answers_path = comparison_dir / "test_comparisons.jsonl"

    # Avoid overwriting results from an earlier comparison.
    if metrics_path.exists() or answers_path.exists():
        raise FileExistsError(
            f"Comparison results already exist in {comparison_dir}. "
            "Choose a new comparison directory before rerunning."
        )

    model.eval()
    model.gradient_checkpointing_disable()
    model.zero_grad(set_to_none=True)

    gc.collect()
    torch.cuda.empty_cache()

    # --------------------------------------------------------
    # 1. TEST LOSS PER DOMAIN
    # --------------------------------------------------------

    per_domain_metrics = {}

    for domain in DOMAINS:
        print(f"\nEvaluating test loss: {domain}")

        test_data = domain_dataset("test", domain)

        if not test_data:
            raise RuntimeError(
                f"No tokenized test examples found for {domain}."
            )

        domain_metrics = {
            "examples": len(test_data),
        }

        for label, adapter_enabled in (
            ("base", False),
            ("adapter", True),
        ):
            gc.collect()
            torch.cuda.empty_cache()

            adapter_context = (
                nullcontext()
                if adapter_enabled
                else model.disable_adapter()
            )

            try:
                with adapter_context:
                    result = evaluation_trainer.evaluate(
                        eval_dataset=test_data,
                        metric_key_prefix=f"{label}_test",
                    )

                domain_metrics[label] = result
                print(label, result)

            except torch.cuda.OutOfMemoryError:
                domain_metrics[label] = {
                    "status": "failed",
                    "reason": "CUDA out of memory",
                }

                print(
                    f"{domain}: {label} evaluation ran out of memory. "
                    "The saved adapter is unaffected."
                )

        per_domain_metrics[domain] = domain_metrics

        # Save after every domain so earlier results survive
        # a later interruption.
        metrics_path.write_text(
            json.dumps(
                per_domain_metrics,
                ensure_ascii=False,
                indent=2,
            ) + "\n",
            encoding="utf-8",
        )

    # --------------------------------------------------------
    # 2. PAIRED ANSWERS: BASE MODEL VS ADAPTER
    # --------------------------------------------------------

    EXAMPLES_PER_DOMAIN = 3
    MAX_NEW_TOKENS = 256

    model.eval()

    gc.collect()
    torch.cuda.empty_cache()

    def generate_answer(example, adapter_enabled):
        """Generate with identical prompts and sampling settings."""
        set_seed(SEED)

        adapter_context = (
            nullcontext()
            if adapter_enabled
            else model.disable_adapter()
        )

        with adapter_context, torch.inference_mode():
            inputs = tokenizer(
                prompt_text(example, tokenizer),
                return_tensors="pt",
                add_special_tokens=False,
            ).to(model.get_input_embeddings().weight.device)

            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,
                top_p=0.8,
                top_k=20,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )

            prompt_length = inputs["input_ids"].shape[1]
            completion_ids = output[
                0, prompt_length:
            ].detach().cpu().tolist()

        return {
            "status": "ok",
            "text": tokenizer.decode(
                completion_ids,
                skip_special_tokens=True,
            ),
            "generated_tokens": len(completion_ids),
            "hit_length_limit": (
                len(completion_ids) == MAX_NEW_TOKENS
                and completion_ids[-1] != tokenizer.eos_token_id
            ),
        }

    comparisons = []

    with answers_path.open("w", encoding="utf-8") as handle:
        for domain in DOMAINS:
            examples = [
                example
                for example in selected_raw["test"]
                if example["domain"] == domain
            ][:EXAMPLES_PER_DOMAIN]

            for example in examples:
                print(f"\nGenerating: {domain} / {example['id']}")

                comparison = {
                    "id": example["id"],
                    "domain": example["domain"],
                    "document_id": example["document_id"],
                    "question": example["question"],
                    "context": example["context"],
                    "reference_answer": example["answer"],
                }

                for label, adapter_enabled in (
                    ("base", False),
                    ("adapter", True),
                ):
                    gc.collect()
                    torch.cuda.empty_cache()

                    try:
                        comparison[label] = generate_answer(
                            example,
                            adapter_enabled=adapter_enabled,
                        )

                    except torch.cuda.OutOfMemoryError:
                        comparison[label] = {
                            "status": "failed",
                            "reason": "CUDA out of memory",
                        }

                        print(
                            f"{label} generation ran out of memory."
                        )

                comparisons.append(comparison)

                # Write each completed pair immediately.
                handle.write(
                    json.dumps(
                        comparison,
                        ensure_ascii=False,
                    ) + "\n"
                )
                handle.flush()

                print("QUESTION:", comparison["question"])
                print(
                    "REFERENCE:",
                    comparison["reference_answer"],
                )

                for label in ("base", "adapter"):
                    result = comparison[label]

                    if result["status"] == "ok":
                        print(f"{label.upper()}:", result["text"])

                        if result["hit_length_limit"]:
                            print(
                                f"{label.upper()}: answer reached "
                                "the generation limit."
                            )
                    else:
                        print(
                            f"{label.upper()}:",
                            result["reason"],
                        )

    print("\nComparison files saved:")
    print("Per-domain test loss:", metrics_path)
    print("Paired answers:", answers_path)
    print("Examples compared:", len(comparisons))


Evaluating test loss: civil
base {'base_test_loss': 1.6399643421173096, 'base_test_model_preparation_time': 0.0048, 'base_test_runtime': 6.6984, 'base_test_samples_per_second': 2.09, 'base_test_steps_per_second': 2.09}
adapter {'adapter_test_loss': 0.9393431544303894, 'adapter_test_model_preparation_time': 0.0048, 'adapter_test_runtime': 8.0589, 'adapter_test_samples_per_second': 1.737, 'adapter_test_steps_per_second': 1.737}

Evaluating test loss: criminal
base {'base_test_loss': 1.2715864181518555, 'base_test_model_preparation_time': 0.0048, 'base_test_runtime': 6.2251, 'base_test_samples_per_second': 2.249, 'base_test_steps_per_second': 2.249}
adapter {'adapter_test_loss': 0.6916986107826233, 'adapter_test_model_preparation_time': 0.0048, 'adapter_test_runtime': 7.4491, 'adapter_test_samples_per_second': 1.879, 'adapter_test_steps_per_second': 1.879}

Evaluating test loss: administrative
base {'base_test_loss': 1.264943242073059, 'base_test_model_preparation_time': 0.0048, 'base_te